# Role Mapping Pipeline

Pipeline ini memproses file **RAW_DATA_SCRAPING_CAPSTONE_.csv** menjadi **DATA_CAPSTONE_MAPPING_AI.csv** dengan langkah:

1. **Load** data mentah dari `data/raw/RAW_DATA_SCRAPING_CAPSTONE_.csv`
2. **Klasifikasi Role** — weighted keyword matching menambah kolom:
   - `role_label`, `label_score`, `label_source`, `label_keyword`
3. **Terjemahan** — kolom `descriptionText` diterjemahkan ke Bahasa Inggris → kolom `translated_descriptionText`
4. **Filter** — hanya baris dengan `descriptionText` berbahasa Inggris (atau berhasil diterjemahkan) yang disimpan
5. **Simpan** hasil akhir ke `data/interim/DATA_CAPSTONE_MAPPING_AI.csv`

---
### Pembobotan Sumber Informasi

| Kolom | Bobot |
|---|---|
| `standardizedTitle` | 5 |
| `title` | 4 |
| `jobFunction` | 2 |
| `industries` | 1 |
| `descriptionText` | 1 |


# Cell 1 — Setup & Konfigurasi Path

In [ ]:
import os

INPUT_FILE        = os.path.join( 'RAW_DATA_SCRAPING_CAPSTONE_.csv')
OUTPUT_FILE       = os.path.join( 'data', 'interim', 'DATA_CAPSTONE_MAPPING_AI.csv')
OUTPUT_ROLE_TABLE = os.path.join( 'data', 'interim', 'tabel_50_role.csv')
CHECKPOINT_FILE   = os.path.join( 'data', 'interim', '_checkpoint_translation.csv')

print('Input  :', INPUT_FILE)
print('Output :', OUTPUT_FILE)
print('Ada?   :', os.path.exists(INPUT_FILE))

# Verifikasi Python yang dipakai
import sys
print('Python :', sys.executable)

Input  : RAW_DATA_SCRAPING_CAPSTONE_.csv
Output : data/interim/DATA_CAPSTONE_MAPPING_AI.csv
Ada?   : True
Python : /usr/bin/python3


# Cell 2 — Load Data Mentah


In [ ]:
import pandas as pd

df = pd.read_csv(
    INPUT_FILE,
    engine='python',
    on_bad_lines='skip',
    quotechar='"',
    escapechar='\\'
)

print('Total baris terbaca :', len(df))
print('Kolom               :', df.columns.tolist())
df.head(3)

Total baris terbaca : 18814
Kolom               : ['applicantsCount', 'applyMethod', 'companyName', 'country', 'descriptionText', 'employmentType', 'industries', 'jobFunction', 'location', 'postedAt', 'salary', 'seniorityLevel', 'standardizedTitle', 'title', 'workRemoteAllowed']


,applicantsCount,applyMethod,companyName,country,descriptionText,employmentType,industries,jobFunction,location,postedAt,salary,seniorityLevel,standardizedTitle,title,workRemoteAllowed
0,25,OffsiteApply,PT IKONSULTAN INOVATAMA,ID,Minimum Requirement\n\nA minimum of a Bachelor...,Full-time,Information Technology & Services,Engineering and Information Technology,"Gambir, Jakarta, Indonesia",2025-02-03T09:21:50.000Z,NaN,Not Applicable,Java Software Engineer,Java Developer,False
1,200,OffsiteApply,Monee,ID,About The Team\n\nAbout the Team\n\nThe Paymen...,Full-time,Financial Services,Engineering and Information Technology,"Jakarta, Indonesia",2026-04-22T13:12:08.000Z,NaN,Entry level,Back End Developer,Back End Engineer (ShopeePay) - Sea Labs,False
2,29,ComplexOnsiteApply,Glints,ID,The ideal candidate will be responsible for de...,Full-time,Financial Services and IT Services and IT Cons...,Information Technology,"Batam, Riau Islands, Indonesia",2026-05-04T04:40:51.000Z,NaN,Mid-Senior level,Frontend Developer,Senior Frontend Developer,False


# Cell 3 — Definisi Kata Kunci per Role (50 roles)

In [ ]:
ROLE_KEYWORDS = {
    "Data Analyst": [
        "data analyst", "business data analyst", "marketing data analyst",
        "financial data analyst", "analytics analyst", "insights analyst",
        "commercial analyst"
    ],
    "Data Scientist": [
        "data scientist", "data science", "predictive model",
        "statistical model", "advanced analytics"
    ],
    "Data Engineer": [
        "data engineer", "data engineering", "etl", "elt",
        "data pipeline", "data warehouse", "data lake", "big data"
    ],
    "Machine Learning Engineer": [
        "machine learning engineer", "ml engineer", "mlops",
        "ai/ml engineer", "model deployment", "ml pipeline"
    ],
    "AI Engineer": [
        "ai engineer", "artificial intelligence", "generative ai",
        "llm", "rag", "ai researcher", "ai specialist"
    ],
    "Business Intelligence Analyst": [
        "business intelligence", "bi analyst", "bi developer",
        "power bi", "tableau", "looker", "dashboard"
    ],
    "Data Architect": [
        "data architect", "analytics architect", "big data migration architect",
        "data architecture", "lakehouse"
    ],
    "Data Governance Specialist": [
        "data governance", "data management", "data steward",
        "data quality", "master data", "metadata"
    ],
    "Database Administrator": [
        "database administrator", "dba", "database admin"
    ],
    "Backend Developer": [
        "backend", "back end", "server-side", "java developer",
        "spring boot", "golang", "node.js"
    ],
    "Frontend Developer": [
        "frontend", "front end", "react", "angular", "vue"
    ],
    "Full Stack Developer": [
        "full stack", "fullstack"
    ],
    "Mobile Developer": [
        "mobile developer", "android", "ios developer", "flutter",
        "react native", "kotlin", "swift"
    ],
    "Software Engineer": [
        "software engineer", "software developer", "developer",
        "programmer", "application engineer"
    ],
    "DevOps Engineer": [
        "devops", "ci/cd", "kubernetes", "docker", "terraform"
    ],
    "Cloud Engineer": [
        "cloud engineer", "cloud infrastructure", "aws", "azure",
        "gcp", "google cloud", "cloud platform"
    ],
    "Site Reliability Engineer": [
        "site reliability", "sre", "reliability engineer"
    ],
    "Cyber Security Analyst": [
        "cyber security", "cybersecurity", "security analyst",
        "soc analyst", "threat analyst", "siem"
    ],
    "Security Engineer": [
        "security engineer", "network security", "application security",
        "cloud security", "devsecops"
    ],
    "Penetration Tester": [
        "penetration tester", "pentest", "ethical hacker",
        "vulnerability assessment"
    ],
    "Network Engineer": [
        "network engineer", "network administrator", "cisco",
        "routing", "switching"
    ],
    "System Administrator": [
        "system administrator", "sysadmin", "system admin",
        "linux administrator"
    ],
    "QA Engineer": [
        "qa engineer", "quality assurance", "software tester",
        "test engineer", "automation testing"
    ],
    "IT Support Specialist": [
        "it support", "technical support", "helpdesk",
        "service desk", "desktop support"
    ],
    "Business Analyst": [
        "business analyst", "system analyst", "requirement analyst",
        "process analyst"
    ],
    "Product Manager": [
        "product manager", "product owner", "product management"
    ],
    "Project Manager": [
        "project manager", "program manager", "scrum master",
        "delivery manager"
    ],
    "UI/UX Designer": [
        "ui/ux", "ux designer", "ui designer", "product designer",
        "user experience", "user interface"
    ],
    "Solutions Architect": [
        "solutions architect", "solution architect", "technical architect",
        "presales consultant", "pre sales solution"
    ],
    "Enterprise Architect": [
        "enterprise architect"
    ],
    "IT Consultant": [
        "it consultant", "technology consultant", "digital consultant",
        "consulting", "consultant"
    ],
    "ERP Consultant": [
        "erp", "sap", "oracle erp", "dynamics", "odoo"
    ],
    "Blockchain Developer": [
        "blockchain", "web3", "smart contract", "solidity"
    ],
    "Game Developer": [
        "game developer", "unity", "unreal", "gamedev"
    ],
    "Embedded/IoT Engineer": [
        "embedded", "iot", "firmware", "microcontroller"
    ],
    "Robotics Engineer": [
        "robotics", "robotic", "automation engineer"
    ],
    "Technical Writer": [
        "technical writer", "documentation specialist",
        "content writer", "copywriter"
    ],
    "Digital Marketing Specialist": [
        "digital marketing", "seo", "sem", "performance marketing",
        "social media", "marketing specialist"
    ],
    "Sales Executive": [
        "sales", "account executive", "business development",
        "sales executive"
    ],
    "Customer Service": [
        "customer service", "customer support", "customer success",
        "call center"
    ],
    "Human Resources Specialist": [
        "human resources", "hr ", "recruiter",
        "talent acquisition", "people analytics"
    ],
    "Finance Analyst": [
        "finance analyst", "financial analyst", "accounting",
        "fp&a", "finance"
    ],
    "Risk & Compliance Analyst": [
        "risk", "compliance", "audit", "governance",
        "regulatory", "fraud analyst", "anti fraud"
    ],
    "Operations Specialist": [
        "operations", "operation specialist", "supply chain", "logistics"
    ],
    "Researcher": [
        "researcher", "research assistant", "research scientist"
    ],
    "Lecturer/Trainer": [
        "lecturer", "teacher", "instructor", "trainer", "faculty"
    ],
    "Legal Specialist": [
        "legal", "lawyer", "paralegal", "contract specialist"
    ],
    "Healthcare Professional": [
        "doctor", "nurse", "pharmacist", "healthcare",
        "medical", "clinical"
    ],
    "Design/Creative Specialist": [
        "graphic designer", "visual designer", "creative",
        "video editor", "motion graphic"
    ],
    "General IT Specialist": [
        "information technology", "it staff", "it specialist", "technology"
    ]
}

print(f'Total role yang didefinisikan: {len(ROLE_KEYWORDS)}')
df.head(3)

Total role yang didefinisikan: 50


,applicantsCount,applyMethod,companyName,country,descriptionText,employmentType,industries,jobFunction,location,postedAt,salary,seniorityLevel,standardizedTitle,title,workRemoteAllowed
0,25,OffsiteApply,PT IKONSULTAN INOVATAMA,ID,Minimum Requirement\n\nA minimum of a Bachelor...,Full-time,Information Technology & Services,Engineering and Information Technology,"Gambir, Jakarta, Indonesia",2025-02-03T09:21:50.000Z,NaN,Not Applicable,Java Software Engineer,Java Developer,False
1,200,OffsiteApply,Monee,ID,About The Team\n\nAbout the Team\n\nThe Paymen...,Full-time,Financial Services,Engineering and Information Technology,"Jakarta, Indonesia",2026-04-22T13:12:08.000Z,NaN,Entry level,Back End Developer,Back End Engineer (ShopeePay) - Sea Labs,False
2,29,ComplexOnsiteApply,Glints,ID,The ideal candidate will be responsible for de...,Full-time,Financial Services and IT Services and IT Cons...,Information Technology,"Batam, Riau Islands, Indonesia",2026-05-04T04:40:51.000Z,NaN,Mid-Senior level,Frontend Developer,Senior Frontend Developer,False


# Cell 4 — Fungsi Klasifikasi Role (Weighted Keyword Matching)


In [ ]:
def norm(value):
    """Normalisasi teks: lowercase, strip, handle NaN."""
    if pd.isna(value):
        return ""
    return str(value).lower()


def classify_role(row):
    """
    Klasifikasi role berdasarkan weighted keyword matching.

    Pembobotan:
      standardizedTitle : 5
      title             : 4
      jobFunction       : 2
      industries        : 1
      descriptionText   : 1  (hanya 1200 karakter pertama)

    Returns pd.Series dengan kolom:
      role_label, label_score, label_source, label_keyword
    """
    standardized_title = norm(row.get("standardizedTitle"))
    title              = norm(row.get("title"))
    job_function       = norm(row.get("jobFunction"))
    industries         = norm(row.get("industries"))
    description        = norm(row.get("descriptionText"))[:1200]

    # (teks, bobot, nama_sumber)
    fields = [
        (standardized_title, 5, "standardizedTitle"),
        (title,              4, "title"),
        (job_function,       2, "jobFunction"),
        (industries,         1, "industries"),
        (description,        1, "descriptionText"),
    ]

    best_role    = "General IT Specialist"
    best_score   = -1
    best_source  = "fallback"
    best_keyword = "fallback_no_match"

    for role, keywords in ROLE_KEYWORDS.items():
        score      = 0
        hit_source  = ""
        hit_keyword = ""

        for text, weight, source in fields:
            for keyword in keywords:
                if keyword in text:
                    score += weight
                    if hit_source == "":
                        hit_source  = source
                        hit_keyword = keyword
                    break  # satu keyword per field sudah cukup

        if score > best_score:
            best_role    = role
            best_score   = score
            best_source  = hit_source
            best_keyword = hit_keyword

    # Jika tidak ada kecocokan → fallback default
    if best_score <= 0:
        best_role    = "General IT Specialist"
        best_score   = 0
        best_source  = "fallback"
        best_keyword = "fallback_no_match"

    return pd.Series({
        "role_label"   : best_role,
        "label_score"  : best_score,
        "label_source" : best_source,
        "label_keyword": best_keyword,
    })


print('Fungsi classify_role siap.')

Fungsi classify_role siap.


# Cell 5 — Jalankan Klasifikasi Role


In [ ]:
print('Mengklasifikasi role... (mungkin beberapa menit untuk data besar)')
labels = df.apply(classify_role, axis=1)

df_labeled = pd.concat([df, labels], axis=1)

print(f'Total baris setelah labeling : {len(df_labeled)}')
print(f'Total baris terlabel         : {df_labeled["role_label"].notna().sum()}')
print()
print('Distribusi top-10 role:')
print(df_labeled['role_label'].value_counts().head(10))

df_labeled[['standardizedTitle','title','role_label','label_score','label_source','label_keyword']].head(5)

Mengklasifikasi role... (mungkin beberapa menit untuk data besar)
Total baris setelah labeling : 18814
Total baris terlabel         : 18814

Distribusi top-10 role:
role_label
Software Engineer        5023
General IT Specialist    1938
Full Stack Developer     1528
AI Engineer               820
QA Engineer               762
Frontend Developer        631
Sales Executive           613
Backend Developer         567
Operations Specialist     542
IT Consultant             542
Name: count, dtype: int64


,standardizedTitle,title,role_label,label_score,label_source,label_keyword
0,Java Software Engineer,Java Developer,Software Engineer,10,standardizedTitle,software engineer
1,Back End Developer,Back End Engineer (ShopeePay) - Sea Labs,Backend Developer,10,standardizedTitle,back end
2,Frontend Developer,Senior Frontend Developer,Frontend Developer,10,standardizedTitle,frontend
3,Information Technology System Analyst,IT System Analyst,Business Analyst,10,standardizedTitle,system analyst
4,PHP Developer,PHP Developer (Wordpress),Software Engineer,10,standardizedTitle,developer


# Cell 6 — Terjemahan descriptionText ke Bahasa Inggris


- Deteksi bahasa setiap baris dengan langdetect
- Jika sudah Inggris ('en') -> salin langsung ke - translated_descriptionText   
- Jika bahasa lain -> terjemahkan dengan deep-translator (GoogleTranslator)
- Gunakan checkpoint (append per-baris) agar bisa dilanjutkan jika terputus


In [ ]:
# ============================================================
# Cell 6 — Terjemahan descriptionText ke Bahasa Inggris
#
# Strategi:
#   - Deteksi bahasa setiap baris dengan langdetect
#   - Jika sudah Inggris ('en') -> salin langsung ke translated_descriptionText
#   - Jika bahasa lain -> terjemahkan dengan deep-translator (GoogleTranslator)
#   - Gunakan checkpoint (append per-baris) agar bisa dilanjutkan jika terputus
# ============================================================

import os, csv, time
import pandas as pd
from langdetect import detect, DetectorFactory
from deep_translator import GoogleTranslator

# Seed agar deteksi konsisten
DetectorFactory.seed = 0


def detect_lang(text):
    """Deteksi bahasa; kembalikan 'unknown' jika teks kosong/error."""
    if not text or str(text).strip() == "" or str(text).lower() == 'nan':
        return 'unknown'
    try:
        return detect(str(text)[:100])
    except Exception:
        return 'error'


def run_translation_pipeline(df_input, checkpoint_file, target_col='descriptionText'):
    """
    Menambah kolom 'translated_descriptionText' ke df_input.
    Mendukung resume dari checkpoint jika proses sebelumnya terputus.
    """
    # Pastikan direktori tujuan ada
    checkpoint_dir = os.path.dirname(checkpoint_file)
    if checkpoint_dir and not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir, exist_ok=True)
        print(f"Direktori dibuat: {checkpoint_dir}")

    translator = GoogleTranslator(source='auto', target='en')

    # Tentukan titik mulai (resume)
    if os.path.exists(checkpoint_file):
        df_done = pd.read_csv(checkpoint_file, engine='python', on_bad_lines='skip')
        start_idx = len(df_done)
        print(f'Resume dari baris ke-{start_idx} (checkpoint ditemukan).')
    else:
        start_idx = 0
        print('Mulai dari awal (tidak ada checkpoint).')

    total = len(df_input)
    print(f'Sisa baris yang perlu diproses: {total - start_idx} dari {total}')

    for i in range(start_idx, total):
        row = df_input.iloc[i].copy()
        original_text = str(row[target_col])

        detected = detect_lang(original_text)

        if detected == 'en':
            # Sudah Inggris - salin langsung
            row['translated_descriptionText'] = original_text
            wait_time = 0.0
        else:
            # Bahasa lain - terjemahkan
            try:
                translated = translator.translate(original_text[:4500])
                row['translated_descriptionText'] = translated if translated else original_text
                wait_time = 1.0
            except Exception:
                row['translated_descriptionText'] = original_text
                wait_time = 2.0

        # Simpan baris ke checkpoint (append)
        pd.DataFrame([row]).to_csv(
            checkpoint_file,
            mode='a',
            index=False,
            header=(not os.path.exists(checkpoint_file)),
            quoting=csv.QUOTE_ALL,
            escapechar='\\',
        )

        if wait_time > 0:
            time.sleep(wait_time)

        if (i + 1) % 500 == 0:
            print(f'  Progres: {i+1}/{total} baris selesai...')

    print(f'Terjemahan selesai! Total {total} baris.')


# Jalankan pipeline terjemahan
run_translation_pipeline(df_labeled, CHECKPOINT_FILE)

Direktori dibuat: data/interim
Mulai dari awal (tidak ada checkpoint).
Sisa baris yang perlu diproses: 18814 dari 18814
  Progres: 500/18814 baris selesai...
  Progres: 1000/18814 baris selesai...
  Progres: 1500/18814 baris selesai...
  Progres: 2000/18814 baris selesai...
  Progres: 2500/18814 baris selesai...
  Progres: 3000/18814 baris selesai...
  Progres: 3500/18814 baris selesai...
  Progres: 4000/18814 baris selesai...
  Progres: 4500/18814 baris selesai...
  Progres: 5000/18814 baris selesai...
  Progres: 5500/18814 baris selesai...
  Progres: 6000/18814 baris selesai...
  Progres: 6500/18814 baris selesai...
  Progres: 7000/18814 baris selesai...
  Progres: 7500/18814 baris selesai...
  Progres: 8000/18814 baris selesai...
  Progres: 8500/18814 baris selesai...
  Progres: 9000/18814 baris selesai...
  Progres: 9500/18814 baris selesai...
  Progres: 10000/18814 baris selesai...
  Progres: 10500/18814 baris selesai...
  Progres: 11000/18814 baris selesai...
  Progres: 11500/188

# Cell 7 — Load Hasil Terjemahan & Filter Bahasa Inggris


 Setelah terjemahan selesai:
   1. Baca checkpoint
   2. Deteksi bahasa translated_descriptionText
   3. Filter: hanya simpan baris yang berhasil menjadi Bahasa Inggris ('en')
   4. Simpan ke DATA_CAPSTONE_MAPPING_AI.csv

In [ ]:
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

# 1. Baca hasil checkpoint
df_translated = pd.read_csv(
    CHECKPOINT_FILE,
    engine='python',
    on_bad_lines='skip'
)
print(f'Baris setelah terjemahan : {len(df_translated)}')


# 2. Deteksi bahasa
def get_lang(text):
    if pd.isna(text) or str(text).strip() == "":
        return "unknown"
    try:
        return detect(str(text)[:200])
    except Exception:
        return "error"

print('Mendeteksi bahasa hasil terjemahan...')
df_translated['_detected_lang'] = df_translated['translated_descriptionText'].apply(get_lang)

print('\nDistribusi bahasa terdeteksi:')
print(df_translated['_detected_lang'].value_counts())


# 3. Filter hanya Bahasa Inggris
df_final = df_translated[df_translated['_detected_lang'] == 'en'].copy()
df_final = df_final.drop(columns=['_detected_lang'])
df_final = df_final.reset_index(drop=True)

print(f'\nBaris awal            : {len(df_translated)}')
print(f'Baris non-Inggris     : {len(df_translated) - len(df_final)}')
print(f'Baris final (Inggris) : {len(df_final)}')

# Preview
df_final[['title','role_label','label_score','label_source','label_keyword','translated_descriptionText']].head(5)

Baris setelah terjemahan : 18814
Mendeteksi bahasa hasil terjemahan...

Distribusi bahasa terdeteksi:
_detected_lang
en       18733
ja          46
id          16
fr          15
zh-cn        2
de           2
Name: count, dtype: int64

Baris awal            : 18814
Baris non-Inggris     : 81
Baris final (Inggris) : 18733


,title,role_label,label_score,label_source,label_keyword,translated_descriptionText
0,Java Developer,Software Engineer,10,standardizedTitle,software engineer,Minimum Requirement\n\nA minimum of a Bachelor...
1,Back End Engineer (ShopeePay) - Sea Labs,Backend Developer,10,standardizedTitle,back end,About The Team\n\nAbout the Team\n\nThe Paymen...
2,Senior Frontend Developer,Frontend Developer,10,standardizedTitle,frontend,The ideal candidate will be responsible for de...
3,IT System Analyst,Business Analyst,10,standardizedTitle,system analyst,INTIKOM is hiring for System Analyst - Full WF...
4,PHP Developer (Wordpress),Software Engineer,10,standardizedTitle,developer,We are seeking a skilled and self-motivated PH...


# Cell 9 — Simpan Output Final


In [ ]:
# Simpan DATA_CAPSTONE_MAPPING_AI.csv
df_final.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f'[SAVED] {OUTPUT_FILE}')
print(f'        {len(df_final)} baris | {len(df_final.columns)} kolom')
print()

# Buat tabel ringkasan role
role_table = pd.DataFrame({
    'role_id'   : [f'R{str(i+1).zfill(2)}' for i in range(len(ROLE_KEYWORDS))],
    'role_label': list(ROLE_KEYWORDS.keys()),
    'keywords'  : ['; '.join(v) for v in ROLE_KEYWORDS.values()]
})

summary = df_final['role_label'].value_counts().reset_index()
summary.columns = ['role_label', 'total_rows']
role_table = role_table.merge(summary, on='role_label', how='left')
role_table['total_rows'] = role_table['total_rows'].fillna(0).astype(int)

role_table.to_csv(OUTPUT_ROLE_TABLE, index=False)
print(f'[SAVED] {OUTPUT_ROLE_TABLE}')
print()
print('Top 20 Role:')
print(role_table.sort_values('total_rows', ascending=False).head(20).to_string(index=False))

[SAVED] data/interim/DATA_CAPSTONE_MAPPING_AI.csv
        18733 baris | 20 kolom

[SAVED] data/interim/tabel_50_role.csv

Top 20 Role:
role_id                    role_label                                                                                                                                     keywords  total_rows
    R14             Software Engineer                                                           software engineer; software developer; developer; programmer; application engineer        5010
    R50         General IT Specialist                                                                                  information technology; it staff; it specialist; technology        1915
    R12          Full Stack Developer                                                                                                                        full stack; fullstack        1525
    R05                   AI Engineer                                                  ai engineer; a

# Cell 10 — Pembersihan Checkpoint & Statistik Akhir


In [ ]:
# Hapus file checkpoint sementara
if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)
    print(f'[CLEANED] Checkpoint dihapus: {CHECKPOINT_FILE}')

# Distribusi role
role_counts = df_final['role_label'].value_counts().reset_index()
role_counts.columns = ['role_label', 'jumlah_data']

print()
print('=== STATISTIK AKHIR ===')
print(f'Total baris output  : {len(df_final)}')
print(f'Total kolom output  : {len(df_final.columns)}')
print(f'File output         : {OUTPUT_FILE}')
print()
print('Distribusi role:')
print(role_counts.to_string(index=False))

[CLEANED] Checkpoint dihapus: data/interim/_checkpoint_translation.csv

=== STATISTIK AKHIR ===
Total baris output  : 18733
Total kolom output  : 20
File output         : data/interim/DATA_CAPSTONE_MAPPING_AI.csv

Distribusi role:
                   role_label  jumlah_data
            Software Engineer         5010
        General IT Specialist         1915
         Full Stack Developer         1525
                  AI Engineer          816
                  QA Engineer          758
           Frontend Developer          627
              Sales Executive          609
            Backend Developer          560
                IT Consultant          540
        Operations Specialist          535
               Data Scientist          431
                 Data Analyst          415
                Data Engineer          407
              DevOps Engineer          367
             Mobile Developer          279
    Risk & Compliance Analyst          238
Business Intelligence Analyst         